# BirdCLEF 2024: MLP Head Submission
This notebook uses the **Residual MLP Head** trained on Day 5. It runs on **Perch v2** embeddings and handles the full 182-species vocabulary.

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import torch
import torch.nn as nn
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

INPUT_DIR = Path('/kaggle/input')
TEST_AUDIO_DIR = INPUT_DIR / 'competitions/birdclef-2024/test_soundscapes'

test_files = list(TEST_AUDIO_DIR.glob('*.ogg'))
if not test_files:
    TEST_AUDIO_DIR = INPUT_DIR / 'competitions/birdclef-2024/unlabeled_soundscapes'
    test_files = list(TEST_AUDIO_DIR.glob('*.ogg'))[:2]

train_df = pd.read_csv(INPUT_DIR / 'competitions/birdclef-2024/train_metadata.csv')
TARGET_SPECIES = sorted(train_df['primary_label'].unique())
N_CLASSES = len(TARGET_SPECIES)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Robust Perch v2 Loading

In [ ]:
try:
    pb_path = next(INPUT_DIR.rglob('saved_model.pb'))
    perch_model = tf.saved_model.load(str(pb_path.parent))
    if 'serving_default' in perch_model.signatures:
        perch_fn = perch_model.signatures['serving_default']
    elif hasattr(perch_model, 'infer_tf'):
        perch_fn = perch_model.infer_tf
    else:
        perch_fn = perch_model
    print("Perch v2 loaded successfully.")
except Exception as e:
    print(f"Perch loading error: {e}")
    perch_fn = None

def get_embeddings(audio_chunk):
    # Ensure chunk is 32kHz and wrapped in a batch dimension
    x = tf.constant(audio_chunk[None, :], dtype=tf.float32)
    
    try:
        # 1. Try positional argument (Standard for most TF versions)
        out = perch_fn(x)
    except Exception:
        try:
            # 2. Try keyword 'inputs' (Common for Perch v2 'serving_default')
            out = perch_fn(inputs=x)
        except Exception:
            # 3. Try keyword 'input' (Some older Perch exports)
            out = perch_fn(input=x)
            
    # Parse the output (it might be a dict or a raw tensor)
    if isinstance(out, dict):
        if 'embedding' in out: return out['embedding'].numpy()[0]
        if 'label' in out: return out['label'].numpy()[0] # Fallback
    return out.numpy()[0]


## 2. MLP Head Definition & Weight Loading

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x): return x + self.net(x)

class MLPHead(nn.Module):
    def __init__(self, in_dim=1536, hidden_dim=512, n_classes=182, dropout=0.3):
        super().__init__()
        self.input_layer = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.res_block = ResidualBlock(hidden_dim, dropout)
        self.output_layer = nn.Linear(hidden_dim, n_classes)
        self.temperature = nn.Parameter(torch.ones(n_classes))
    def forward(self, x, apply_calibration=True):
        x = self.input_layer(x)
        x = self.res_block(x)
        logits = self.output_layer(x)
        if apply_calibration:
            t = torch.nn.functional.softplus(self.temperature) + 1e-4
            logits = logits / t
        return logits

model = MLPHead(n_classes=N_CLASSES).to(DEVICE)
# --- Fixed Weight Loading ---
# --- Failsafe Weight Loading ---
import glob

# Search for any file named 'mlp_head_final.pth' anywhere in /kaggle/input
potential_paths = glob.glob('/kaggle/input/**/mlp_head_final.pth', recursive=True)

if potential_paths:
    actual_path = potential_paths[0]
    try:
        model.load_state_dict(torch.load(actual_path, map_location=DEVICE))
        model.eval()
        print(f"✅ SUCCESS: Automatically found and loaded weights from: {actual_path}")
    except Exception as e:
        print(f"❌ Error loading MLP weights: {e}")
else:
    print("❌ STILL NOT FOUND. Listing all files in /kaggle/input to help you:")
    for root, dirs, files in os.walk('/kaggle/input'):
        for file in files:
            if file.endswith('.pth'):
                print(os.path.join(root, file))



## 3. Inference Loop

In [ ]:
# --- OPTIMIZED BATCHED INFERENCE ---
model = model.to('cpu') # Keep MLP on CPU to save VRAM
model.eval()

all_preds = []
for file_path in tqdm(test_files):
    audio_id = file_path.stem
    try:
        y, _ = sf.read(file_path, dtype='float32')
        if len(y.shape) > 1: y = y.mean(axis=1)
        
        # 1. PREPARE ALL CHUNKS FOR THE FILE
        chunks = []
        time_ends = []
        for i in range(0, len(y), CHUNK_LEN):
            time_end = ((i // CHUNK_LEN) + 1) * 5
            if time_end > 240: break
            chunk = y[i : i + CHUNK_LEN]
            if len(chunk) < CHUNK_LEN: chunk = np.pad(chunk, (0, CHUNK_LEN - len(chunk)))
            chunks.append(chunk)
            time_ends.append(time_end)
        
        # 2. BATCH INFERENCE (Perch on GPU)
        # We send the entire file (48 chunks) as a single batch to the GPU
        batch_x = tf.constant(np.array(chunks), dtype=tf.float32)
                # --- Fixed Perch Call ---
        if 'serving_default' in str(perch_fn):
            out = perch_fn(inputs=batch_x)
        else:
            out = perch_fn(batch_x)

        
        if isinstance(out, dict):
            embs = out['embedding'].numpy()
        else:
            embs = out.numpy()
            
        # 3. MLP HEAD (on CPU)
        with torch.no_grad():
            x_torch = torch.tensor(embs) # Process the whole batch of embeddings at once
            probs = torch.sigmoid(model(x_torch)).numpy()
            
        # 4. STORE RESULTS
        for j, time_end in enumerate(time_ends):
            row = {'row_id': f"{audio_id}_{time_end}"}
            for k, sp in enumerate(TARGET_SPECIES):
                row[sp] = probs[j, k]
            all_preds.append(row)
            
        # Clear TF memory every file to prevent the 9-minute OOM crash
        tf.keras.backend.clear_session()
            
    except Exception as e:
        print(f"Error on {audio_id}: {e}")
        for t in range(5, 245, 5):
            all_preds.append({'row_id': f"{audio_id}_{t}", **{sp: 0.0 for sp in TARGET_SPECIES}})

# Final Save
sub_df = pd.DataFrame(all_preds)
sub_df.to_csv('submission.csv', index=False)
